# Proyecto 3 - Analisis de Datos para Simulacion de Eventos Discretos

## Objetivo del analisis
Este notebook transforma una base historica de operacion agil en parametros cuantitativos para construir y validar un modelo de simulacion de eventos discretos en FlexSim.

## Proposito dentro del proyecto
A partir del archivo `base_agile_sin_bloqueos.xlsx`, hoja `Datos`, se estiman y documentan:
- patrones de llegada de tareas;
- tiempos de proceso en desarrollo, QA y retrabajo;
- probabilidades de eventos discretos relevantes;
- capacidades promedio de recursos;
- metricas historicas para validar el modelo.

## Estructura del notebook
- Las secciones 1 a 4 cubren carga, descripcion de variables y validacion basica de la base.
- La seccion 3 concentra la descripcion de columnas para evitar duplicar el diccionario de datos en varias partes.
- Las secciones 5 a 20 desarrollan el analisis estadistico y operacional de las variables relevantes.
- Las secciones 21 a 23 consolidan parametros, validacion y escenarios para FlexSim.

## Consideraciones importantes
- La base esta agregada por dia; por ello, el tiempo entre llegadas se aproxima usando el promedio de tareas nuevas por dia.
- Requisitos y diseno no tienen tiempos separados en el Excel; cualquier tiempo para esas etapas tendria que declararse como supuesto.
- `Lead_Time_Horas`, `Cycle_Time_Horas`, `WIP` y `Tareas_Completadas` se conservan principalmente para contraste y validacion del modelo, no como entradas directas.


## 1. Importación de librerías
En esta sección se cargan las librerías necesarias para lectura de datos, análisis estadístico, ajuste de distribuciones y generación de gráficas. También se configura la presentación de tablas para mejorar la lectura del notebook.

In [ ]:
import os
import warnings
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

plt.style.use("ggplot")
OUTPUT_DIR = Path("graficas_analisis")
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Carga del archivo Excel
Aquí se valida que el archivo exista, se intenta cargar la hoja `Datos` y se verifica que las columnas mínimas requeridas estén presentes. Esta validación es importante porque todo el análisis posterior depende de la estructura esperada.

In [ ]:
archivo_excel = Path("base_agile_sin_bloqueos.xlsx")
hoja_excel = "Datos"

columnas_esperadas = [
    "Fecha", "Tareas_Nuevas", "Tipo_Tarea", "Prioridad", "Proyecto", "Complejidad",
    "Bugs_QA", "Cambio_Requisitos", "Hotfix_Urgente", "Saturacion_QA", "Ausencia_Recurso",
    "Repriorizacion_Backlog", "Retrabajo_Flag", "Horas_Retrabajo", "Developers_Disponibles",
    "QA_Disponibles", "Horas_Extra", "Tiempo_Desarrollo_Horas", "Tiempo_QA_Horas",
    "Cantidad_Bugs", "Tareas_Completadas", "Sprint", "WIP", "Lead_Time_Horas", "Cycle_Time_Horas"
]

if not archivo_excel.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo '{archivo_excel}'. Colóquelo en la misma carpeta del notebook."
    )

try:
    df = pd.read_excel(archivo_excel, sheet_name=hoja_excel)
except ValueError as exc:
    raise ValueError(f"No fue posible leer la hoja '{hoja_excel}'. Verifique su nombre.") from exc

columnas_faltantes = [col for col in columnas_esperadas if col not in df.columns]
if columnas_faltantes:
    raise ValueError(
        "Faltan columnas obligatorias en el Excel: " + ", ".join(columnas_faltantes)
    )

print("Archivo cargado correctamente.")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print("\nPrimeras filas:")
display(df.head())
print("\nNombres de columnas:")
display(pd.DataFrame({"Columna": df.columns}))
print("\nTipos de datos detectados:")
display(df.dtypes.rename("dtype").to_frame())

## 3. Descripción de las columnas
La siguiente tabla documenta el significado operativo de cada variable, su utilidad analítica y su posible uso posterior dentro del modelo de simulación. Esta trazabilidad es importante porque queremos que cada parámetro de FlexSim quede respaldado por una fuente de datos o por un supuesto explícito.

In [ ]:
descripcion_columnas = pd.DataFrame([
    ["Fecha", "Fecha del registro diario", "Permite análisis temporal, tendencias y comparación demanda-capacidad", "Índice temporal del sistema"],
    ["Tareas_Nuevas", "Cantidad de tareas que entran al sistema en el día", "Sirve para estimar tasa de llegada y tiempo promedio entre llegadas", "Configuración del Source o patrón de arribos"],
    ["Tipo_Tarea", "Clasificación de la tarea: Feature, Bug, Refactor, Hotfix", "Permite obtener probabilidades de mezcla de trabajo", "Asignación de labels o rutas por tipo"],
    ["Prioridad", "Nivel de prioridad de cada día o de las tareas entrantes", "Permite estimar reglas de atención o despacho", "Reglas de prioridad o colas con disciplina diferenciada"],
    ["Proyecto", "Proyecto al que pertenecen las tareas", "Mide concentración de la demanda por proyecto", "Label para segmentación o escenarios multicliente"],
    ["Complejidad", "Nivel de complejidad de la carga", "Permite caracterizar heterogeneidad de trabajo", "Label descriptivo o modificador de tiempos si se justifica"],
    ["Bugs_QA", "Indicador binario de aparición de bugs en QA", "Estima probabilidad de falla en control de calidad", "Decisión probabilística de retrabajo o reproceso"],
    ["Cambio_Requisitos", "Indicador de cambio de requisitos", "Estima frecuencia de cambios del cliente o del backlog", "Bifurcación hacia replanificación o retrabajo"],
    ["Hotfix_Urgente", "Indicador de trabajo urgente correctivo", "Cuantifica interrupciones prioritarias", "Ruta urgente o preempción"],
    ["Saturacion_QA", "Indicador de saturación en QA", "Ayuda a evaluar presión sobre QA", "Puede modelarse como evento o usarse solo como validación"],
    ["Ausencia_Recurso", "Indicador de ausencia de personal", "Mide disrupciones por capacidad variable", "Reducción temporal de recursos"],
    ["Repriorizacion_Backlog", "Indicador de reordenamiento del backlog", "Mide cambios de secuencia de trabajo", "Cambio en reglas de prioridad"],
    ["Retrabajo_Flag", "Indicador binario de retrabajo", "Estima probabilidad de reproceso", "Enviar entidades a una etapa adicional"],
    ["Horas_Retrabajo", "Horas adicionales asociadas al retrabajo", "Permite estimar tiempo de reproceso", "Tiempo adicional si ocurre retrabajo"],
    ["Developers_Disponibles", "Número de desarrolladores disponibles", "Permite estimar capacidad de desarrollo", "Capacidad del recurso de desarrollo"],
    ["QA_Disponibles", "Número de analistas QA disponibles", "Permite estimar capacidad de QA", "Capacidad del recurso de QA"],
    ["Horas_Extra", "Horas extra trabajadas", "Indica presión de capacidad y respuesta operativa", "Escenarios de capacidad ampliada"],
    ["Tiempo_Desarrollo_Horas", "Tiempo invertido en desarrollo", "Se usa para modelar tiempo de servicio de desarrollo", "Process time en estación de desarrollo"],
    ["Tiempo_QA_Horas", "Tiempo invertido en QA", "Se usa para modelar tiempo de servicio de QA", "Process time en estación QA"],
    ["Cantidad_Bugs", "Número de bugs encontrados", "Permite validar calidad y relación con retrabajo", "Métrica de salida y validación"],
    ["Tareas_Completadas", "Cantidad de tareas terminadas en el día", "Permite contrastar demanda vs salida y validar throughput", "Métrica de validación de producción"],
    ["Sprint", "Identificador del sprint", "Permite segmentar análisis por iteraciones", "Agrupación para escenarios o calendarios"],
    ["WIP", "Trabajo en proceso", "Sirve para evaluar congestión del sistema", "Métrica de validación del inventario en proceso"],
    ["Lead_Time_Horas", "Tiempo total desde llegada hasta entrega", "Métrica integral de desempeño del sistema", "Validación del tiempo total"],
    ["Cycle_Time_Horas", "Tiempo desde inicio de trabajo hasta finalización", "Métrica del tiempo efectivo del flujo activo", "Validación del tiempo de proceso neto"],
], columns=["Columna", "Significado", "Uso en el análisis", "Posible uso en FlexSim"])

display(descripcion_columnas)

## 4. Limpieza y validación de datos
En esta sección se revisa la calidad de la información. El análisis convierte `Fecha` a tipo fecha, identifica nulos, duplicados y valores negativos en variables donde no deberían existir. El objetivo es documentar si la base está lista para análisis o si requiere tratamiento adicional.

In [ ]:
df_analisis = df.copy()

df_analisis["Fecha"] = pd.to_datetime(df_analisis["Fecha"], errors="coerce")

columnas_flags = [
    "Bugs_QA", "Cambio_Requisitos", "Hotfix_Urgente", "Saturacion_QA",
    "Ausencia_Recurso", "Repriorizacion_Backlog", "Retrabajo_Flag"
]

columnas_numericas = [
    "Tareas_Nuevas", "Complejidad", "Bugs_QA", "Cambio_Requisitos", "Hotfix_Urgente",
    "Saturacion_QA", "Ausencia_Recurso", "Repriorizacion_Backlog", "Retrabajo_Flag",
    "Horas_Retrabajo", "Developers_Disponibles", "QA_Disponibles", "Horas_Extra",
    "Tiempo_Desarrollo_Horas", "Tiempo_QA_Horas", "Cantidad_Bugs", "Tareas_Completadas",
    "WIP", "Lead_Time_Horas", "Cycle_Time_Horas"
]

columnas_numericas_descriptivas = [
    col for col in columnas_numericas if col not in columnas_flags
]

for col in columnas_numericas:
    df_analisis[col] = pd.to_numeric(df_analisis[col], errors="coerce")

nulos = df_analisis.isna().sum().rename("Valores nulos").to_frame()
duplicados = int(df_analisis.duplicated().sum())

columnas_no_negativas = [
    "Tareas_Nuevas", "Complejidad", "Horas_Retrabajo", "Developers_Disponibles", "QA_Disponibles",
    "Horas_Extra", "Tiempo_Desarrollo_Horas", "Tiempo_QA_Horas", "Cantidad_Bugs",
    "Tareas_Completadas", "WIP", "Lead_Time_Horas", "Cycle_Time_Horas"
]

resumen_negativos = []
for col in columnas_no_negativas:
    cantidad_negativos = int((df_analisis[col] < 0).fillna(False).sum())
    resumen_negativos.append({"Columna": col, "Valores negativos": cantidad_negativos})
resumen_negativos = pd.DataFrame(resumen_negativos)

resumen_calidad = pd.DataFrame([
    ["Filas totales", df_analisis.shape[0]],
    ["Columnas totales", df_analisis.shape[1]],
    ["Fechas inválidas", int(df_analisis["Fecha"].isna().sum())],
    ["Registros duplicados", duplicados],
    ["Columnas con nulos", int((nulos["Valores nulos"] > 0).sum())],
    ["Columnas con negativos no esperados", int((resumen_negativos["Valores negativos"] > 0).sum())],
], columns=["Indicador", "Valor"])

print("Resumen de calidad de datos")
display(resumen_calidad)
print("\nValores nulos por columna")
display(nulos)
print("\nValores negativos no esperados")
display(resumen_negativos)

if (
    int(df_analisis["Fecha"].isna().sum()) == 0
    and duplicados == 0
    and int((nulos["Valores nulos"] > 0).sum()) == 0
    and int((resumen_negativos["Valores negativos"] > 0).sum()) == 0
):
    print("Conclusión: la base está lista para análisis sin observaciones de calidad evidentes.")
else:
    print("Conclusión: la base presenta alertas de calidad; podemos continuar el análisis, pero interpretaremos los resultados con cautela.")

## 5. Estadisticos descriptivos generales
En esta seccion se calculan estadisticos descriptivos solo para variables numericas continuas o de conteo relevantes para el modelo.

**No se incluyen las variables binarias tipo flag**: `Bugs_QA`, `Cambio_Requisitos`, `Hotfix_Urgente`, `Saturacion_QA`, `Ausencia_Recurso`, `Repriorizacion_Backlog` y `Retrabajo_Flag`.

La razon es que, para esas variables, la interpretacion correcta no es la dispersion sino la **probabilidad de ocurrencia**. Por eso se analizan mas adelante en la seccion de probabilidades de eventos.


In [ ]:
resumen_estadistico = df_analisis[columnas_numericas_descriptivas].describe(percentiles=[0.25, 0.50, 0.75]).T
resumen_estadistico = resumen_estadistico[["mean", "50%", "std", "min", "25%", "75%", "max"]]
resumen_estadistico = resumen_estadistico.rename(columns={
    "mean": "Media",
    "50%": "Mediana",
    "std": "Desv_Estandar",
    "min": "Minimo",
    "25%": "P25",
    "75%": "P75",
    "max": "Maximo"
})
resumen_estadistico.to_excel("resumen_estadistico.xlsx")
display(resumen_estadistico)

### Gráficas de apoyo
Las siguientes gráficas complementan la tabla descriptiva con una vista rápida de la forma, dispersión y presencia de posibles valores atípicos en algunas variables clave del sistema.

In [ ]:
variables_graficas = [
    "Tareas_Nuevas",
    "Tiempo_Desarrollo_Horas",
    "Tiempo_QA_Horas",
    "Horas_Retrabajo",
    "Tareas_Completadas",
    "WIP",
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, columna in zip(axes, variables_graficas):
    serie = df_analisis[columna].dropna()
    ax.hist(serie, bins=min(12, max(5, serie.nunique())), color="#4C78A8", edgecolor="black", alpha=0.85)
    ax.set_title(f"Histograma de {columna}")
    ax.set_xlabel(columna)
    ax.set_ylabel("Frecuencia")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "estadisticos_histogramas_clave.png", dpi=200)
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, columna in zip(axes, variables_graficas):
    serie = df_analisis[columna].dropna()
    ax.boxplot(serie, vert=True, patch_artist=True, boxprops=dict(facecolor="#F58518", alpha=0.75))
    ax.set_title(f"Boxplot de {columna}")
    ax.set_ylabel(columna)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "estadisticos_boxplots_clave.png", dpi=200)
plt.show()

**Interpretación:** esta tabla resume la magnitud y variabilidad de cada variable numérica. Más adelante se utilizarán algunas de estas medidas para estimar distribuciones y construir parámetros de entrada o validación para FlexSim.

## 6. Análisis de llegadas de tareas
La columna `Tareas_Nuevas` aproxima la demanda diaria que entra al sistema. Como la base está agregada por día y no contiene marcas de tiempo exactas de cada llegada, el tiempo entre llegadas se aproxima usando el promedio diario y la expresión `24 / promedio_tareas_nuevas_por_dia`.

In [ ]:
serie_llegadas = df_analisis["Tareas_Nuevas"].dropna()

promedio_tareas_nuevas = serie_llegadas.mean()
desv_tareas_nuevas = serie_llegadas.std()
min_tareas_nuevas = serie_llegadas.min()
max_tareas_nuevas = serie_llegadas.max()
tasa_promedio_llegada = promedio_tareas_nuevas
media_tiempo_entre_llegadas = np.nan if promedio_tareas_nuevas <= 0 else 24 / promedio_tareas_nuevas

resumen_llegadas = pd.DataFrame([
    ["Promedio tareas nuevas por día", promedio_tareas_nuevas],
    ["Desviación estándar", desv_tareas_nuevas],
    ["Mínimo", min_tareas_nuevas],
    ["Máximo", max_tareas_nuevas],
    ["Tasa promedio de llegada (tareas/día)", tasa_promedio_llegada],
    ["Tiempo promedio entre llegadas (horas)", media_tiempo_entre_llegadas],
], columns=["Indicador", "Valor"])

display(resumen_llegadas)

plt.figure(figsize=(9, 5))
plt.hist(serie_llegadas, bins=min(12, max(5, serie_llegadas.nunique())), edgecolor="black")
plt.title("Histograma de tareas nuevas por día")
plt.xlabel("Tareas nuevas")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "hist_tareas_nuevas.png", dpi=200)
plt.show()

expresion_llegadas_flexsim = f"exponential(0, {media_tiempo_entre_llegadas:.4f}, getstream(current))"
print("Parámetro recomendado para FlexSim:")
print(expresion_llegadas_flexsim)

**Interpretación:** la expresión anterior representa una aproximación de llegadas para FlexSim. La leemos con cautela porque no proviene de tiempos individuales entre arribos, sino de una agregación diaria. Si después conseguimos timestamps por tarea, recalibraremos esta parte del modelo.

## 7. Frecuencias y probabilidades de tipos de tarea
Esta sección caracteriza la mezcla de trabajo que entra al sistema. La distribución de `Tipo_Tarea` es relevante porque permite asignar labels, definir rutas o introducir lógica diferenciada por clase de tarea dentro del modelo.

In [ ]:
def tabla_frecuencias(df_base, columna, nombre_probabilidad="Probabilidad"):
    frecuencias = df_base[columna].fillna("No informado").value_counts(dropna=False).rename("Cantidad")
    tabla = frecuencias.to_frame()
    tabla[nombre_probabilidad] = tabla["Cantidad"] / tabla["Cantidad"].sum()
    tabla["Porcentaje"] = tabla[nombre_probabilidad] * 100
    tabla.index.name = columna
    return tabla.reset_index()

frecuencia_tipos = tabla_frecuencias(df_analisis, "Tipo_Tarea")
frecuencia_tipos["Uso en FlexSim"] = "Asignación de label o decisión de ruta"
display(frecuencia_tipos)

plt.figure(figsize=(9, 5))
plt.bar(frecuencia_tipos["Tipo_Tarea"].astype(str), frecuencia_tipos["Cantidad"], edgecolor="black")
plt.title("Frecuencia de tipos de tarea")
plt.xlabel("Tipo de tarea")
plt.ylabel("Cantidad")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "barras_tipo_tarea.png", dpi=200)
plt.show()

prob_tipos = dict(zip(frecuencia_tipos["Tipo_Tarea"], frecuencia_tipos["Probabilidad"]))
for tipo in ["Feature", "Bug", "Refactor", "Hotfix"]:
    print(f"Probabilidad de {tipo}: {prob_tipos.get(tipo, 0):.4f}")

**Ejemplo de uso en FlexSim:** estas probabilidades pueden utilizarse para asignar un label de tipo a cada entidad al momento de creación. Por ejemplo, un bloque de decisión puede comparar un número uniforme contra probabilidades acumuladas para etiquetar la tarea como `Feature`, `Bug`, `Refactor` o `Hotfix`.

## 8. Frecuencias y probabilidades de prioridades
La prioridad permite modelar reglas de atención, secuenciación o urgencia. Su uso en FlexSim depende del diseño del modelo: puede alimentar disciplina de cola, criterios de despacho o rutas especiales.

In [ ]:
frecuencia_prioridad = tabla_frecuencias(df_analisis, "Prioridad")
frecuencia_prioridad["Interpretación"] = "La usaremos para definir reglas de atención, preempción o cola priorizada"
display(frecuencia_prioridad)

plt.figure(figsize=(9, 5))
plt.bar(frecuencia_prioridad["Prioridad"].astype(str), frecuencia_prioridad["Cantidad"], edgecolor="black")
plt.title("Frecuencia de prioridades")
plt.xlabel("Prioridad")
plt.ylabel("Cantidad")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "barras_prioridad.png", dpi=200)
plt.show()

**Interpretación:** la prioridad `Media` concentra la mayor proporción de registros (45.21%), pero `Alta`, `Baja` y `Critica` también tienen presencia relevante. Por eso no vamos a simplificar la prioridad a una sola categoría dominante; la representaremos explícitamente en la lógica de atención de FlexSim.

## 9. Análisis de proyectos
Este análisis permite verificar si la carga histórica está balanceada entre proyectos o si uno de ellos domina la operación. Esa concentración importa porque puede justificar escenarios por portafolio o segmentos de demanda.

In [ ]:
frecuencia_proyecto = tabla_frecuencias(df_analisis, "Proyecto")
display(frecuencia_proyecto)

plt.figure(figsize=(10, 5))
plt.bar(frecuencia_proyecto["Proyecto"].astype(str), frecuencia_proyecto["Cantidad"], edgecolor="black")
plt.title("Distribución de tareas por proyecto")
plt.xlabel("Proyecto")
plt.ylabel("Cantidad")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "barras_proyecto.png", dpi=200)
plt.show()

proyecto_dominante = frecuencia_proyecto.iloc[0]
print(
    f"Proyecto dominante: {proyecto_dominante['Proyecto']} con {proyecto_dominante['Porcentaje']:.2f}% de los registros."
)

**Interpretación:** la distribución por proyecto es relativamente homogénea (`Data` 27.67%, `Web` 26.85%, `Backend` 23.56% y `Mobile` 21.92%). Eso nos da soporte para trabajar con un modelo agregado sin que un solo proyecto sesgue por completo el comportamiento del sistema.

## 10. Análisis de complejidad
La complejidad permite describir la heterogeneidad de la demanda. Podemos utilizarla solo como atributo descriptivo o como modificador de tiempos, pero esta segunda decisión la sustentaremos con evidencia o con una política explícita del modelo.

In [ ]:
serie_complejidad = df_analisis["Complejidad"].dropna()
resumen_complejidad = pd.DataFrame([
    ["Promedio", serie_complejidad.mean()],
    ["Desviación estándar", serie_complejidad.std()],
    ["Mínimo", serie_complejidad.min()],
    ["Máximo", serie_complejidad.max()],
], columns=["Indicador", "Valor"])
frecuencia_complejidad = tabla_frecuencias(df_analisis, "Complejidad")

display(resumen_complejidad)
display(frecuencia_complejidad)

plt.figure(figsize=(9, 5))
plt.hist(serie_complejidad, bins=min(12, max(5, serie_complejidad.nunique())), edgecolor="black")
plt.title("Histograma de complejidad")
plt.xlabel("Complejidad")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "hist_complejidad.png", dpi=200)
plt.show()

expresion_complejidad = f"duniform({serie_complejidad.min():.0f}, {serie_complejidad.max():.0f}, getstream(current))"
print("Rango recomendado para FlexSim:")
print(expresion_complejidad)

**Interpretación:** por ahora tratamos la complejidad como un label descriptivo. En la siguiente sección verificamos si realmente tiene relación con la duración antes de decidir si la usamos o no como modificador de tiempos.

## 11. Relacion entre complejidad y duracion
En esta seccion se evalua si la variable `Complejidad` tiene una relacion util con la duracion de las tareas. Para ello se calculan correlaciones de Pearson y Spearman, se comparan los tiempos promedio por nivel de complejidad y se emite un veredicto automatico.


In [ ]:
columnas_relacion_complejidad = [
    "Complejidad", "Tiempo_Desarrollo_Horas", "Tiempo_QA_Horas",
    "Lead_Time_Horas", "Cycle_Time_Horas", "Horas_Retrabajo"
]

pearson_complejidad = df_analisis[columnas_relacion_complejidad].corr(numeric_only=True).loc[
    "Complejidad", ["Tiempo_Desarrollo_Horas", "Tiempo_QA_Horas", "Lead_Time_Horas", "Cycle_Time_Horas", "Horas_Retrabajo"]
]
spearman_complejidad = df_analisis[columnas_relacion_complejidad].corr(method="spearman", numeric_only=True).loc[
    "Complejidad", ["Tiempo_Desarrollo_Horas", "Tiempo_QA_Horas", "Lead_Time_Horas", "Cycle_Time_Horas", "Horas_Retrabajo"]
]

correlaciones_complejidad = pd.DataFrame({
    "Pearson": pearson_complejidad,
    "Spearman": spearman_complejidad,
}).round(4)
display(correlaciones_complejidad)

resumen_complejidad_tiempos = df_analisis.groupby("Complejidad")[["Tiempo_Desarrollo_Horas", "Tiempo_QA_Horas"]].agg(["count", "mean", "median"]).round(4)
display(resumen_complejidad_tiempos)

promedios_por_complejidad = df_analisis.groupby("Complejidad")[["Tiempo_Desarrollo_Horas", "Tiempo_QA_Horas"]].mean().sort_index()
plt.figure(figsize=(10, 5))
plt.plot(promedios_por_complejidad.index, promedios_por_complejidad["Tiempo_Desarrollo_Horas"], marker="o", label="Tiempo_Desarrollo_Horas")
plt.plot(promedios_por_complejidad.index, promedios_por_complejidad["Tiempo_QA_Horas"], marker="s", label="Tiempo_QA_Horas")
plt.title("Promedio de duracion por nivel de complejidad")
plt.xlabel("Complejidad")
plt.ylabel("Horas promedio")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "complejidad_vs_duracion.png", dpi=200)
plt.show()

fuerza_desarrollo = max(abs(correlaciones_complejidad.loc["Tiempo_Desarrollo_Horas", "Pearson"]), abs(correlaciones_complejidad.loc["Tiempo_Desarrollo_Horas", "Spearman"]))
fuerza_qa = max(abs(correlaciones_complejidad.loc["Tiempo_QA_Horas", "Pearson"]), abs(correlaciones_complejidad.loc["Tiempo_QA_Horas", "Spearman"]))

if max(fuerza_desarrollo, fuerza_qa) < 0.20:
    veredicto_complejidad = "No se observa una relacion clara entre complejidad y duracion. Por eso la usaremos como variable descriptiva, no como modificador directo de tiempos."
elif max(fuerza_desarrollo, fuerza_qa) < 0.40:
    veredicto_complejidad = "La relacion entre complejidad y duracion es debil. Solo la usaremos como ajuste secundario si decidimos imponer una regla operativa adicional."
else:
    veredicto_complejidad = "Existe evidencia de una relacion moderada o mayor entre complejidad y duracion. En este caso sí la usaremos para ajustar tiempos del modelo."

print(veredicto_complejidad)


**Interpretacion:** Pearson mide si existe relacion lineal entre complejidad y duracion; Spearman mide si existe una tendencia monotona aunque no sea lineal. Si ambos coeficientes quedan cercanos a cero, no hay evidencia suficiente para sostener que tareas mas complejas duren mas tiempo de manera consistente.


## 12. Probabilidades de eventos discretos
Las variables binarias permiten estimar probabilidades de ocurrencia de eventos que afectan el flujo. Cuando una columna está codificada como `0/1`, su promedio equivale directamente a la probabilidad de ocurrencia del evento.

In [ ]:
columnas_eventos = [
    "Bugs_QA", "Cambio_Requisitos", "Hotfix_Urgente", "Saturacion_QA",
    "Ausencia_Recurso", "Repriorizacion_Backlog", "Retrabajo_Flag"
]

filas_eventos = []
for col in columnas_eventos:
    serie = pd.to_numeric(df_analisis[col], errors="coerce")
    ocurrencias = int((serie == 1).sum())
    no_ocurrencias = int((serie == 0).sum())
    probabilidad = float(serie.mean())
    filas_eventos.append([
        col, ocurrencias, no_ocurrencias, probabilidad, probabilidad * 100,
        "Evento probabilístico o métrica de validación en FlexSim"
    ])

probabilidades_eventos = pd.DataFrame(
    filas_eventos,
    columns=["Evento", "Ocurrencias", "No ocurrencias", "Probabilidad", "Porcentaje", "Uso en FlexSim"]
)
probabilidades_eventos.to_excel("probabilidades_eventos.xlsx", index=False)
display(probabilidades_eventos)

**Interpretación:** estas probabilidades pueden implementarse como decisiones estocásticas en FlexSim. En algunos casos, como `Saturacion_QA`, también es válido usar la variable solo como referencia histórica y dejar que el fenómeno emerja naturalmente de la congestión de la cola.

## 13. Análisis de recursos disponibles
Se estudian los recursos `Developers_Disponibles`, `QA_Disponibles` y `Horas_Extra` para estimar capacidades base. En simulación suele ser razonable usar el promedio redondeado como capacidad inicial del sistema, y luego evaluar escenarios alternativos.

In [ ]:
columnas_recursos = ["Developers_Disponibles", "QA_Disponibles", "Horas_Extra"]
resumen_recursos = df_analisis[columnas_recursos].agg(["mean", "min", "max", "std"]).T
resumen_recursos = resumen_recursos.rename(columns={"mean": "Media", "min": "Minimo", "max": "Maximo", "std": "Desv_Estandar"})
display(resumen_recursos)

for columna in columnas_recursos:
    serie = df_analisis[columna].dropna()
    if columna in ["Developers_Disponibles", "QA_Disponibles"]:
        frecuencias = serie.value_counts().sort_index()
        plt.figure(figsize=(8, 4.5))
        plt.bar(frecuencias.index.astype(str), frecuencias.values, edgecolor="black", color="#4C78A8")
        plt.title(f"Frecuencia de {columna}")
        plt.xlabel(columna)
        plt.ylabel("Cantidad de registros")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f"frecuencia_{columna.lower()}.png", dpi=200)
        plt.show()
    else:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), gridspec_kw={"width_ratios": [4, 1]})
        axes[0].hist(serie, bins=15, edgecolor="black", color="#72B7B2", alpha=0.85)
        axes[0].set_title("Distribucion de Horas_Extra")
        axes[0].set_xlabel("Horas_Extra")
        axes[0].set_ylabel("Frecuencia")
        axes[1].boxplot(serie, vert=True, patch_artist=True, boxprops=dict(facecolor="#F58518", alpha=0.8))
        axes[1].set_title("Boxplot")
        axes[1].set_ylabel("Horas_Extra")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "distribucion_horas_extra.png", dpi=200)
        plt.show()

capacidad_developers = int(round(df_analisis["Developers_Disponibles"].mean()))
capacidad_qa = int(round(df_analisis["QA_Disponibles"].mean()))
print(f"Capacidad recomendada de developers: {capacidad_developers}")
print(f"Capacidad recomendada de QA: {capacidad_qa}")

**Interpretación:** los recursos disponibles se mueven en rangos acotados (`Developers_Disponibles` entre 3 y 7, y `QA_Disponibles` entre 1 y 3). Por eso usaremos el promedio redondeado como capacidad base y dejaremos las variaciones diarias para escenarios posteriores, no para el modelo inicial.

## 14. Análisis de tiempos de proceso
Aquí se analizan los tiempos directamente relacionados con el flujo del sistema. Es importante separar entre variables que se usarán como entrada del modelo (`Tiempo_Desarrollo_Horas`, `Tiempo_QA_Horas`, `Horas_Retrabajo`) y variables que se usarán solo para validación (`Lead_Time_Horas`, `Cycle_Time_Horas`).

In [ ]:
columnas_tiempos_input = ["Tiempo_Desarrollo_Horas", "Tiempo_QA_Horas", "Horas_Retrabajo"]
columnas_tiempos_validacion = ["Lead_Time_Horas", "Cycle_Time_Horas"]
columnas_tiempos = columnas_tiempos_input + columnas_tiempos_validacion

resumen_tiempos = df_analisis[columnas_tiempos].agg(["mean", "std", "min", "max"]).T
resumen_tiempos = resumen_tiempos.rename(columns={"mean": "Media", "std": "Desv_Estandar", "min": "Minimo", "max": "Maximo"})

tabla_tiempos_input = resumen_tiempos.loc[columnas_tiempos_input].copy()
tabla_tiempos_input["Uso en el modelo"] = [
    "Tiempo de servicio en desarrollo",
    "Tiempo de servicio en QA",
    "Tiempo adicional si ocurre retrabajo"
]

tabla_tiempos_validacion = resumen_tiempos.loc[columnas_tiempos_validacion].copy()
tabla_tiempos_validacion["Uso en el modelo"] = [
    "Métrica de validación; no usar como input directo",
    "Métrica de validación; no usar como input directo"
]

print("Tiempos usados como entrada del modelo")
display(tabla_tiempos_input)
print("\nTiempos usados como validación del modelo")
display(tabla_tiempos_validacion)

for columna in columnas_tiempos:
    plt.figure(figsize=(8, 4.5))
    plt.hist(df_analisis[columna].dropna(), bins=12, edgecolor="black")
    plt.title(f"Histograma de {columna}")
    plt.xlabel(columna)
    plt.ylabel("Frecuencia")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"hist_{columna.lower()}.png", dpi=200)
    plt.show()

**Aclaración clave:**
- `Tiempo_Desarrollo_Horas` se usa como tiempo de servicio en desarrollo.
- `Tiempo_QA_Horas` se usa como tiempo de servicio en QA.
- `Horas_Retrabajo` se usa como tiempo adicional si ocurre retrabajo.
- `Lead_Time_Horas` y `Cycle_Time_Horas` no los usaremos como entradas directas, sino como métricas para validar el comportamiento del modelo simulado.

## 15. Análisis condicionado de retrabajo
No basta con observar el promedio global de `Horas_Retrabajo`, porque muchos días pueden no tener retrabajo. Por eso se calcula el comportamiento condicionado a los casos donde `Retrabajo_Flag == 1`.

In [ ]:
retrabajo_condicionado = df_analisis.loc[df_analisis["Retrabajo_Flag"] == 1, "Horas_Retrabajo"].dropna()
promedio_retrabajo_general = df_analisis["Horas_Retrabajo"].mean()
media_retrabajo_condicionado = retrabajo_condicionado.mean()
desv_retrabajo_condicionado = retrabajo_condicionado.std()
min_retrabajo_condicionado = retrabajo_condicionado.min()
max_retrabajo_condicionado = retrabajo_condicionado.max()

resumen_retrabajo = pd.DataFrame([
    ["Promedio general Horas_Retrabajo", promedio_retrabajo_general],
    ["Promedio condicionado", media_retrabajo_condicionado],
    ["Desviación condicionada", desv_retrabajo_condicionado],
    ["Mínimo condicionado", min_retrabajo_condicionado],
    ["Máximo condicionado", max_retrabajo_condicionado],
], columns=["Indicador", "Valor"])

display(resumen_retrabajo)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), gridspec_kw={"width_ratios": [4, 1]})
axes[0].hist(retrabajo_condicionado, bins=12, edgecolor="black", color="#E45756", alpha=0.85)
axes[0].set_title("Horas_Retrabajo condicionado a Retrabajo_Flag = 1")
axes[0].set_xlabel("Horas_Retrabajo")
axes[0].set_ylabel("Frecuencia")
axes[1].boxplot(retrabajo_condicionado, vert=True, patch_artist=True, boxprops=dict(facecolor="#72B7B2", alpha=0.8))
axes[1].set_title("Boxplot")
axes[1].set_ylabel("Horas_Retrabajo")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "retrabajo_condicionado.png", dpi=200)
plt.show()

expresion_retrabajo = (
    f"max(0.1, normal({media_retrabajo_condicionado:.4f}, {0 if pd.isna(desv_retrabajo_condicionado) else desv_retrabajo_condicionado:.4f}, getstream(current)))"
)
print("Parámetro recomendado para FlexSim:")
print(expresion_retrabajo)

**Interpretación:** el promedio condicionado representa mejor el tiempo extra real cuando sí ocurre retrabajo. En esta base la diferencia es grande: `Horas_Retrabajo` tiene un promedio general de 1.251 horas, pero cuando `Retrabajo_Flag = 1` el promedio sube a 5.189 horas. Por eso modelaremos el reproceso con la estimación condicionada.

## 16. Ajuste de distribuciones
En esta sección probamos varias distribuciones candidatas para variables numéricas relevantes. El ajuste se evalúa mediante la prueba Kolmogorov-Smirnov (KS). La distribución seleccionada es una recomendación estadística y la validaremos también con sentido del proceso y coherencia operacional.

In [ ]:
variables_distribucion = [
    "Tareas_Nuevas", "Tiempo_Desarrollo_Horas", "Tiempo_QA_Horas",
    "Horas_Retrabajo", "Lead_Time_Horas", "Cycle_Time_Horas"
]

distribuciones_candidatas = {
    "normal": stats.norm,
    "exponencial": stats.expon,
    "gamma": stats.gamma,
    "lognormal": stats.lognorm,
    "weibull": stats.weibull_min,
    "uniforme": stats.uniform,
}

def ajustar_distribuciones(serie, nombre_variable):
    datos = pd.Series(serie).dropna().astype(float)
    datos = datos[np.isfinite(datos)]
    resultados = []
    if len(datos) < 5:
        return pd.DataFrame(columns=["Variable", "Distribución", "KS", "p_value", "Parámetros"])

    for nombre_dist, dist in distribuciones_candidatas.items():
        try:
            params = dist.fit(datos)
            ks_stat, p_value = stats.kstest(datos, dist.cdf, args=params)
            resultados.append([
                nombre_variable,
                nombre_dist,
                ks_stat,
                p_value,
                ", ".join([f"{p:.6f}" for p in params])
            ])
        except Exception:
            resultados.append([
                nombre_variable,
                nombre_dist,
                np.nan,
                np.nan,
                "No convergió"
            ])
    tabla = pd.DataFrame(resultados, columns=["Variable", "Distribución", "KS", "p_value", "Parámetros"])
    return tabla.sort_values(by="KS", na_position="last")

resultados_ajuste = []
for variable in variables_distribucion:
    resultados_ajuste.append(ajustar_distribuciones(df_analisis[variable], variable))

ajuste_distribuciones_completo = pd.concat(resultados_ajuste, ignore_index=True)
ajuste_distribuciones_completo = ajuste_distribuciones_completo.sort_values(by=["Variable", "KS"], na_position="last")

mejor_distribucion = ajuste_distribuciones_completo.dropna(subset=["KS"]).groupby("Variable", as_index=False).first()
mejor_distribucion.to_excel("ajuste_distribuciones.xlsx", index=False)

print("Mejor distribución por variable")
display(mejor_distribucion)

**Interpretación:** una mejor distribución estadística no siempre es la mejor distribución para el modelo. Si el resultado contradice el comportamiento real del proceso, priorizaremos la lógica operativa y usaremos el ajuste solo como apoyo cuantitativo.

## 17. Correlaciones
Las correlaciones ayudan a identificar relaciones entre demanda, capacidad, calidad y tiempos del sistema. Estas relaciones no prueban causalidad, pero orientan hipótesis para interpretar resultados y construir escenarios.

In [ ]:
variables_correlacion = [
    "Tareas_Nuevas", "Developers_Disponibles", "QA_Disponibles", "Horas_Extra",
    "Tiempo_Desarrollo_Horas", "Tiempo_QA_Horas", "Horas_Retrabajo", "Cantidad_Bugs",
    "Tareas_Completadas", "WIP", "Lead_Time_Horas", "Cycle_Time_Horas"
]

matriz_correlacion = df_analisis[variables_correlacion].corr(numeric_only=True)
display(matriz_correlacion)

plt.figure(figsize=(11, 8))
plt.imshow(matriz_correlacion, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlación")
plt.xticks(range(len(variables_correlacion)), variables_correlacion, rotation=90)
plt.yticks(range(len(variables_correlacion)), variables_correlacion)
plt.title("Matriz de correlación")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "heatmap_correlacion.png", dpi=200)
plt.show()

relaciones_clave = pd.DataFrame([
    ["WIP vs Lead Time", matriz_correlacion.loc["WIP", "Lead_Time_Horas"]],
    ["Cantidad_Bugs vs Horas_Retrabajo", matriz_correlacion.loc["Cantidad_Bugs", "Horas_Retrabajo"]],
    ["QA_Disponibles vs Tiempo_QA_Horas", matriz_correlacion.loc["QA_Disponibles", "Tiempo_QA_Horas"]],
    ["Tareas_Nuevas vs Tareas_Completadas", matriz_correlacion.loc["Tareas_Nuevas", "Tareas_Completadas"]],
], columns=["Relación", "Correlación"])

display(relaciones_clave)

**Interpretación:** revisamos especialmente las relaciones entre `WIP` y `Lead Time`, `Cantidad_Bugs` y `Horas_Retrabajo`, y `QA_Disponibles` y `Tiempo_QA_Horas` porque podrían revelar congestión, reproceso o sensibilidad de capacidad. Sin embargo, en esta base las correlaciones observadas son cercanas a cero, por lo que no encontramos evidencia lineal fuerte para sostener esas asociaciones.

## 18. Comparación demanda vs capacidad de salida
Se compara la entrada diaria de trabajo (`Tareas_Nuevas`) contra la salida diaria (`Tareas_Completadas`). Esta comparación permite identificar si existe acumulación potencial de backlog o si la capacidad histórica fue suficiente en promedio.

In [ ]:
promedio_tareas_completadas = df_analisis["Tareas_Completadas"].mean()
diferencia_promedio = promedio_tareas_nuevas - promedio_tareas_completadas
porcentaje_brecha = np.nan if promedio_tareas_nuevas == 0 else (diferencia_promedio / promedio_tareas_nuevas) * 100

resumen_demanda_capacidad = pd.DataFrame([
    ["Promedio tareas nuevas por día", promedio_tareas_nuevas],
    ["Promedio tareas completadas por día", promedio_tareas_completadas],
    ["Diferencia promedio", diferencia_promedio],
    ["Porcentaje de brecha", porcentaje_brecha],
], columns=["Indicador", "Valor"])

display(resumen_demanda_capacidad)

serie_temporal = df_analisis.sort_values("Fecha")
plt.figure(figsize=(11, 5))
plt.plot(serie_temporal["Fecha"], serie_temporal["Tareas_Nuevas"], label="Tareas_Nuevas")
plt.plot(serie_temporal["Fecha"], serie_temporal["Tareas_Completadas"], label="Tareas_Completadas")
plt.title("Demanda vs capacidad de salida")
plt.xlabel("Fecha")
plt.ylabel("Cantidad")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "demanda_vs_capacidad.png", dpi=200)
plt.show()

if diferencia_promedio > 0:
    conclusion_demanda_capacidad = "En promedio entran más tareas de las que salen; existe acumulación potencial de backlog."
else:
    conclusion_demanda_capacidad = "En promedio salen tantas o más tareas de las que entran; la capacidad histórica luce suficiente."

print(conclusion_demanda_capacidad)

**Interpretación:** esta comparación da una señal clara de presión operativa: en promedio entran 11.570 tareas por día y se completan 8.693, así que la brecha es positiva en 2.877 tareas por día. Con este resultado esperamos que el modelo refleje acumulación de trabajo o mayores tiempos de espera si no se ajusta la capacidad.

## 19. Análisis de WIP
El `WIP` refleja la carga activa del sistema. Es una métrica clave para validar la simulación porque resume el nivel de trabajo simultáneo y la congestión operativa observada históricamente.

In [ ]:
serie_wip = df_analisis["WIP"].dropna()
resumen_wip = pd.DataFrame([
    ["Promedio", serie_wip.mean()],
    ["Desviación estándar", serie_wip.std()],
    ["Mínimo", serie_wip.min()],
    ["Máximo", serie_wip.max()],
], columns=["Indicador", "Valor"])

display(resumen_wip)

plt.figure(figsize=(8, 4.5))
plt.hist(serie_wip, bins=12, edgecolor="black")
plt.title("Histograma de WIP")
plt.xlabel("WIP")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "hist_wip.png", dpi=200)
plt.show()

**Interpretación:** el `WIP` observado varía entre 3 y 14 tareas, con promedio de 8.663. Eso muestra que sí existe acumulación de trabajo en el sistema, pero en esta base no aparece una relación lineal fuerte entre `WIP` y `Lead Time`; por eso tomaremos el `WIP` principalmente como referencia de validación y no como evidencia suficiente de congestión por sí sola.

## 20. Análisis de Lead Time y Cycle Time
`Lead Time` y `Cycle Time` son métricas de desempeño global. Su comparación ayuda a separar tiempo total en el sistema versus tiempo de trabajo efectivo. Si el `Lead Time` es mucho mayor que el `Cycle Time`, el problema suele estar en esperas, colas o backlog.

In [ ]:
resumen_lead_cycle = df_analisis[["Lead_Time_Horas", "Cycle_Time_Horas"]].agg(["mean", "std"]).T
resumen_lead_cycle = resumen_lead_cycle.rename(columns={"mean": "Media", "std": "Desv_Estandar"})
display(resumen_lead_cycle)

plt.figure(figsize=(10, 4.5))
plt.subplot(1, 2, 1)
plt.hist(df_analisis["Lead_Time_Horas"].dropna(), bins=12, edgecolor="black")
plt.title("Lead Time")
plt.xlabel("Horas")
plt.ylabel("Frecuencia")

plt.subplot(1, 2, 2)
plt.hist(df_analisis["Cycle_Time_Horas"].dropna(), bins=12, edgecolor="black")
plt.title("Cycle Time")
plt.xlabel("Horas")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lead_cycle_comparacion.png", dpi=200)
plt.show()

brecha_lead_cycle = df_analisis["Lead_Time_Horas"].mean() - df_analisis["Cycle_Time_Horas"].mean()
print(f"Brecha promedio Lead Time - Cycle Time: {brecha_lead_cycle:.4f} horas")

**Interpretacion:** la brecha promedio entre `Lead Time` y `Cycle Time` es de 0.625 horas, así que en esta base no vemos una separación fuerte entre tiempo total en el sistema y tiempo de trabajo efectivo. Eso sugiere que las esperas existen, pero no dominan el comportamiento de forma marcada en los datos historicos disponibles.

## 21. Parámetros finales para FlexSim
En esta sección se consolidan los principales parámetros recomendados para el modelo. La tabla resume el valor calculado, la distribución o fórmula sugerida, la fuente en Excel y su uso operacional dentro de FlexSim.

In [ ]:
prob_prioridades = dict(zip(frecuencia_prioridad["Prioridad"], frecuencia_prioridad["Probabilidad"]))
prob_eventos = dict(zip(probabilidades_eventos["Evento"], probabilidades_eventos["Probabilidad"]))

parametros_flexsim = pd.DataFrame([
    ["Tiempo promedio entre llegadas", media_tiempo_entre_llegadas, expresion_llegadas_flexsim, "Tareas_Nuevas", "Source_Tareas", "Aproximado por agregación diaria"],
    ["Distribución sugerida de llegadas", promedio_tareas_nuevas, expresion_llegadas_flexsim, "Tareas_Nuevas", "Source_Tareas", "Usar con cautela por no tener timestamps"],
    ["Probabilidad Feature", prob_tipos.get("Feature", 0), "Probabilidad empírica", "Tipo_Tarea", "Asignación de label", "Usar probabilidades acumuladas"],
    ["Probabilidad Bug", prob_tipos.get("Bug", 0), "Probabilidad empírica", "Tipo_Tarea", "Asignación de label", "Usar probabilidades acumuladas"],
    ["Probabilidad Refactor", prob_tipos.get("Refactor", 0), "Probabilidad empírica", "Tipo_Tarea", "Asignación de label", "Usar probabilidades acumuladas"],
    ["Probabilidad Hotfix", prob_tipos.get("Hotfix", 0), "Probabilidad empírica", "Tipo_Tarea", "Asignación de label", "Usar probabilidades acumuladas"],
    ["Probabilidades de prioridad", str(prob_prioridades), "Probabilidad empírica", "Prioridad", "Cola o despacho por prioridad", "Justificar política operativa"],
    ["Probabilidad de bugs QA", prob_eventos.get("Bugs_QA", np.nan), "Probabilidad empírica", "Bugs_QA", "Decisión o validación", "Puede conectar con retrabajo"],
    ["Probabilidad de cambio de requisitos", prob_eventos.get("Cambio_Requisitos", np.nan), "Probabilidad empírica", "Cambio_Requisitos", "Bifurcación o replanificación", "Evento discreto"],
    ["Probabilidad de hotfix urgente", prob_eventos.get("Hotfix_Urgente", np.nan), "Probabilidad empírica", "Hotfix_Urgente", "Ruta urgente o prioridad alta", "Evento discreto"],
    ["Probabilidad de saturación QA", prob_eventos.get("Saturacion_QA", np.nan), "Probabilidad empírica", "Saturacion_QA", "Validación o evento", "Preferible como resultado emergente"],
    ["Probabilidad de ausencia de recurso", prob_eventos.get("Ausencia_Recurso", np.nan), "Probabilidad empírica", "Ausencia_Recurso", "Reducción temporal de capacidad", "Evento discreto"],
    ["Probabilidad de repriorización", prob_eventos.get("Repriorizacion_Backlog", np.nan), "Probabilidad empírica", "Repriorizacion_Backlog", "Cambio en cola o secuencia", "Evento discreto"],
    ["Probabilidad de retrabajo", prob_eventos.get("Retrabajo_Flag", np.nan), "Probabilidad empírica", "Retrabajo_Flag", "Reproceso", "Usar con tiempo condicionado"],
    ["Capacidad promedio developers", capacidad_developers, f"Capacity = {capacidad_developers}", "Developers_Disponibles", "Recurso desarrollo", "Promedio redondeado"],
    ["Capacidad promedio QA", capacidad_qa, f"Capacity = {capacidad_qa}", "QA_Disponibles", "Recurso QA", "Promedio redondeado"],
    ["Tiempo desarrollo", df_analisis["Tiempo_Desarrollo_Horas"].mean(), f"max(0.1, normal({df_analisis['Tiempo_Desarrollo_Horas'].mean():.4f}, {df_analisis['Tiempo_Desarrollo_Horas'].std():.4f}, getstream(current)))", "Tiempo_Desarrollo_Horas", "Process desarrollo", "Validar contra ajuste de distribuciones"],
    ["Tiempo QA", df_analisis["Tiempo_QA_Horas"].mean(), f"max(0.1, normal({df_analisis['Tiempo_QA_Horas'].mean():.4f}, {df_analisis['Tiempo_QA_Horas'].std():.4f}, getstream(current)))", "Tiempo_QA_Horas", "Process QA", "Validar contra ajuste de distribuciones"],
    ["Tiempo retrabajo condicionado", media_retrabajo_condicionado, expresion_retrabajo, "Horas_Retrabajo", "Process retrabajo", "Condicionado a Retrabajo_Flag == 1"],
    ["WIP promedio", df_analisis["WIP"].mean(), "Valor histórico", "WIP", "Validación", "No usar como input directo"],
    ["Lead Time promedio", df_analisis["Lead_Time_Horas"].mean(), "Valor histórico", "Lead_Time_Horas", "Validación", "No usar como input directo"],
    ["Cycle Time promedio", df_analisis["Cycle_Time_Horas"].mean(), "Valor histórico", "Cycle_Time_Horas", "Validación", "No usar como input directo"],
    ["Tareas completadas promedio", promedio_tareas_completadas, "Valor histórico", "Tareas_Completadas", "Validación throughput", "No usar como input directo"],
], columns=["Parametro", "Valor", "Distribución o fórmula recomendada", "Fuente en Excel", "Uso en FlexSim", "Observación"])

parametros_flexsim.to_excel("parametros_flexsim.xlsx", index=False)
display(parametros_flexsim)

## 22. Validación del modelo
Después de construir el modelo en FlexSim, compararemos estas métricas históricas contra los resultados simulados. El objetivo no es replicar exactamente cada día, sino aproximar de forma consistente el desempeño promedio y la variabilidad observada.

In [ ]:
metricas_validacion = pd.DataFrame([
    ["Tareas completadas por día", promedio_tareas_completadas, "Comparar throughput promedio y dispersión"],
    ["WIP promedio", df_analisis["WIP"].mean(), "Comparar nivel de congestión del sistema"],
    ["Lead Time promedio", df_analisis["Lead_Time_Horas"].mean(), "Comparar tiempo total de permanencia"],
    ["Cycle Time promedio", df_analisis["Cycle_Time_Horas"].mean(), "Comparar tiempo de flujo activo"],
    ["Saturación QA", prob_eventos.get("Saturacion_QA", np.nan), "Comparar presión sobre la etapa de QA"],
    ["Cantidad de bugs", df_analisis["Cantidad_Bugs"].mean(), "Comparar nivel de defectos"],
    ["Retrabajo", prob_eventos.get("Retrabajo_Flag", np.nan), "Comparar frecuencia de reproceso"],
], columns=["Metrica", "Valor histórico", "Uso para validación"])

metricas_validacion.to_excel("metricas_validacion.xlsx", index=False)
display(metricas_validacion)

## 23. Exportación de resultados
El notebook exporta automáticamente los principales resultados en archivos Excel y guarda las gráficas en la carpeta `graficas_analisis`. Esto facilita reutilizar los parámetros en FlexSim y documentar el trabajo para la entrega académica.

In [ ]:
archivos_generados = pd.DataFrame([
    ["resumen_estadistico.xlsx", Path("resumen_estadistico.xlsx").exists()],
    ["probabilidades_eventos.xlsx", Path("probabilidades_eventos.xlsx").exists()],
    ["ajuste_distribuciones.xlsx", Path("ajuste_distribuciones.xlsx").exists()],
    ["parametros_flexsim.xlsx", Path("parametros_flexsim.xlsx").exists()],
    ["metricas_validacion.xlsx", Path("metricas_validacion.xlsx").exists()],
    ["graficas_analisis/", OUTPUT_DIR.exists()],
], columns=["Archivo o carpeta", "Disponible"])

display(archivos_generados)
print("Las tablas principales se exportan en formato Excel y las gráficas se guardan como PNG en la carpeta 'graficas_analisis'.")

## 24. Conclusiones por sección
A continuación sintetizamos una conclusión puntual por cada sección del notebook, desde la `#1` hasta la `#23`, para dejar un cierre compacto del análisis realizado.

1. **Importación de librerías.** La configuración inicial fue suficiente para cargar, analizar, graficar y exportar los resultados sin depender de herramientas adicionales fuera del entorno del proyecto.
2. **Carga del archivo Excel.** La base `base_agile_sin_bloqueos.xlsx` pudo leerse correctamente desde la hoja `Datos`, por lo que contamos con la estructura mínima necesaria para desarrollar el análisis.
3. **Descripción de las columnas.** Las variables del archivo quedaron trazadas frente a su significado operativo y a su posible papel dentro del modelo, lo que evita supuestos implícitos al momento de parametrizar FlexSim.
4. **Limpieza y validación de datos.** La base quedó convertida a formatos numéricos útiles para análisis y, si bien pueden existir alertas puntuales, el conjunto es utilizable para construir parámetros y métricas históricas.
5. **Estadísticos descriptivos generales.** Las variables numéricas no binarias muestran la magnitud y dispersión del sistema, mientras que las flags quedaron separadas porque su lectura correcta es probabilística, no descriptiva.
6. **Análisis de llegadas de tareas.** La llegada se aproximó con el promedio diario de tareas nuevas, lo que nos da un parámetro de entrada útil aunque agregado, adecuado para una primera versión del modelo.
7. **Frecuencias y probabilidades de tipos de tarea.** La mezcla histórica de `Feature`, `Bug`, `Refactor` y `Hotfix` nos permite representar la composición real de la demanda en la generación de entidades.
8. **Frecuencias y probabilidades de prioridades.** Como `Media` domina pero no anula a las demás categorías, modelaremos la prioridad explícitamente en vez de reducirla a una sola clase dominante.
9. **Análisis de proyectos.** La distribución entre `Data`, `Web`, `Backend` y `Mobile` es lo bastante balanceada como para justificar un modelo agregado sin depender de un único proyecto.
10. **Análisis de complejidad.** La complejidad describe heterogeneidad de la demanda, pero por sí sola no basta para convertirla en un modificador directo de tiempos.
11. **Relación entre complejidad y duración.** Las correlaciones de Pearson y Spearman quedaron cercanas a cero, así que no encontramos evidencia suficiente para usar la complejidad como predictor fuerte de duración.
12. **Probabilidades de eventos discretos.** Las variables binarias permitieron estimar probabilidades empíricas directamente utilizables para cambios de flujo, retrabajo, hotfixes y otros eventos del sistema.
13. **Análisis de recursos disponibles.** Los recursos se mueven en rangos acotados, por lo que usaremos capacidades base redondeadas y dejaremos las variaciones diarias como sensibilidad o escenarios posteriores.
14. **Análisis de tiempos de proceso.** `Tiempo_Desarrollo_Horas`, `Tiempo_QA_Horas` y `Horas_Retrabajo` quedaron identificados como entradas naturales del modelo, mientras que `Lead Time` y `Cycle Time` se reservaron para validación.
15. **Análisis condicionado de retrabajo.** El retrabajo debe modelarse condicionado a `Retrabajo_Flag = 1`, porque el tiempo medio cuando sí ocurre es mucho más representativo que el promedio global.
16. **Ajuste de distribuciones.** Para cada variable relevante identificamos la distribución con mejor ajuste estadístico y la dejamos exportada como referencia para la parametrización del modelo.
17. **Correlaciones.** No encontramos relaciones lineales fuertes entre las combinaciones clave revisadas, así que estas asociaciones no bastan por sí solas para sostener hipótesis fuertes de congestión o reproceso.
18. **Comparación demanda vs capacidad de salida.** La demanda promedio supera claramente a las tareas completadas, lo que indica presión operativa y sugiere acumulación de trabajo si la capacidad no cambia.
19. **Análisis de WIP.** El `WIP` confirma que existe acumulación de trabajo en el sistema, aunque no aparece una relación lineal marcada con `Lead Time` en esta base histórica.
20. **Análisis de Lead Time y Cycle Time.** La brecha promedio entre ambas métricas existe, pero no es lo bastante grande como para concluir que las esperas dominen por completo el comportamiento del sistema.
21. **Parámetros finales para FlexSim.** El notebook consolidó en una sola tabla los parámetros de llegadas, tiempos, probabilidades, capacidades y métricas de referencia necesarios para llevar el análisis al modelo.
22. **Validación del modelo.** Definimos un conjunto concreto de métricas históricas contra las que compararemos la simulación para juzgar si reproduce razonablemente el comportamiento observado.
23. **Exportación de resultados.** El análisis termina dejando tablas y gráficas reutilizables, lo que facilita documentar el trabajo y trasladar los resultados al montaje del modelo en FlexSim.
